# Write And Reload Waveform JSON

This notebook writes waveform configuration JSON files and reloads one of them through the repo-local waveform API.

## 1. Setup

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Find the Hardware Capture repo root from a notebook or script working directory."""

    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        redstone_dir = candidate / "hardware_capture" / "rf" / "redstone"
        if (redstone_dir / "sequence.py").exists() and (redstone_dir / "waveform.py").exists():
            return candidate
    raise FileNotFoundError("Could not find the Hardware Capture repo root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REDSTONE_DIR = REPO_ROOT / "hardware_capture" / "rf" / "redstone"
EXAMPLES_DIR = REDSTONE_DIR / "examples"
OUTPUT_DIR = EXAMPLES_DIR / "generated"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Output dir:", OUTPUT_DIR)

from hardware_capture.rf.redstone import Chirp, Sine, load_waveform_json

## 2. Build Waveforms

In [ ]:
chirp = Chirp(
    points=5120,
    sweep_width_hz=5000.0,
    dwell_s=5.0e-6,
    offset_hz=2500.0,
    phase=0.0,
    phase_steps=1024,
)

sine = Sine(
    points=64,
    amp_min=0.0,
    amp_max=1.0,
    phase_min=0.0,
    phase_max=180.0,
    cycles=1.0,
    phase_steps=1024,
)

{
    "chirp": chirp.summary().to_dict(),
    "sine": sine.summary().to_dict(),
}

## 3. Write JSON Files

In [ ]:
chirp_path = chirp.write_json(OUTPUT_DIR / "chirp_waveform.json")
sine_path = sine.write_json(
    OUTPUT_DIR / "sine_waveform_with_summary.json",
    include_summary=True,
)

print("Wrote compact chirp JSON:", chirp_path)
print("Wrote sine JSON with summary:", sine_path)

## 4. Reload And Inspect

In [ ]:
loaded = load_waveform_json(chirp_path)
phase_rows = loaded.phase_column(precision=3)

{
    "loaded_type": loaded.waveform_type,
    "phase_rows": len(phase_rows or []),
    "first_10": (phase_rows or [])[:10],
}